### 1. Exploratory Data Analysis (EDA)

This notebook builds and evaluates a CNN image classifier. It scans the dataset, assigns labels using the filename rules below, displays example images, and reports the actual class distribution.

**Important:** Verify the filename-to-label rules against your dataset before training. The rules are assumptions based on filename patterns, not medical ground truth.

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Google Colab setup. If you use another Jupyter environment, comment out
# the next two lines and set DATASET_DIR to your dataset folder.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DATASET_DIR = "/content/drive/MyDrive/Split_smol"

extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp",
              "*.JPG", "*.JPEG", "*.PNG", "*.BMP", "*.WEBP")
image_paths = []
for ext in extensions:
    image_paths.extend(
        glob.glob(os.path.join(DATASET_DIR, "**", ext), recursive=True)
    )
image_paths = sorted(set(image_paths))

print(f"Total images found: {len(image_paths)}")
if not image_paths:
    raise FileNotFoundError(
        f"No supported images found under {DATASET_DIR}. "
        "Check the folder path and make sure the dataset is available."
    )

df = pd.DataFrame({"path": image_paths})
df["filename"] = df["path"].apply(os.path.basename)

def guess_label(filename):
    """Assign a class using the project's filename naming conventions."""
    fn = filename.lower()
    if fn.startswith("isic_"):
        return "ISIC_Lesion"
    if any(term in fn for term in ("ringworm", "tinea", "daad")):
        return "Ringworm"
    if fn.startswith(("1_", "2_")):
        return "Numbered_Class"
    return "Other_Infection"

df["label"] = df["filename"].apply(guess_label)

print("\nClass distribution (estimated from filenames):")
display(df["label"].value_counts().rename_axis("class").to_frame("image_count"))

if df["label"].nunique() < 2:
    raise ValueError(
        "At least two classes are required. Check guess_label() and the dataset filenames."
    )

# Basic image validation: report unreadable files before TensorFlow training.
bad_files = []
for image_path in df["path"]:
    try:
        with Image.open(image_path) as image:
            image.verify()
    except Exception:
        bad_files.append(image_path)

if bad_files:
    print(f"\nSkipping {len(bad_files)} unreadable image(s). First examples:")
    for path in bad_files[:10]:
        print(path)
    df = df[~df["path"].isin(bad_files)].reset_index(drop=True)

if df.empty or df["label"].nunique() < 2:
    raise ValueError("Not enough valid images/classes remain after image validation.")

print(f"\nValid images available: {len(df)}")


In [ ]:
# Display one sample image per class
unique_labels = sorted(df["label"].unique())
fig, axes = plt.subplots(
    1, len(unique_labels),
    figsize=(4 * len(unique_labels), 4),
    squeeze=False
)

for i, label in enumerate(unique_labels):
    sample_path = df.loc[df["label"] == label, "path"].iloc[0]
    with Image.open(sample_path) as image:
        axes[0, i].imshow(image.convert("RGB"))
        axes[0, i].set_title(f"{label}\n{image.size}")
        axes[0, i].axis("off")

plt.tight_layout()
plt.show()


### 2. CNN Model Implementation

We split the images into training and validation sets, resize them to 128 × 128 pixels, and train a custom CNN. Labels are encoded consistently using `label_map`.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Rescaling
)
from sklearn.model_selection import train_test_split

tf.random.set_seed(SEED)

# Stratified splitting requires at least two images in every class.
counts = df["label"].value_counts()
if counts.min() < 2:
    raise ValueError(
        "Every class needs at least 2 valid images for a stratified split. "
        f"Class counts: {counts.to_dict()}"
    )

IMG_HEIGHT, IMG_WIDTH = 128, 128
BATCH_SIZE = 32
VALIDATION_FRACTION = 0.20

label_map = {
    label: index
    for index, label in enumerate(sorted(df["label"].unique()))
}
num_classes = len(label_map)

# Ensure the validation set can contain at least one image from every class.
validation_count = int(np.ceil(len(df) * VALIDATION_FRACTION))
if validation_count < num_classes:
    raise ValueError(
        "The validation split is too small to include every class. "
        "Add more images or increase VALIDATION_FRACTION."
    )

train_df, val_df = train_test_split(
    df,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=df["label"]
)

keys_tensor = tf.constant(list(label_map.keys()), dtype=tf.string)
vals_tensor = tf.constant(list(label_map.values()), dtype=tf.int64)
label_table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor),
    default_value=-1
)

def load_and_preprocess_image(path, label_name):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(
        raw, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    label = label_table.lookup(label_name)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["path"].values, train_df["label"].values)
)
train_ds = (
    train_ds
    .shuffle(len(train_df), seed=SEED, reshuffle_each_iteration=True)
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["path"].values, val_df["label"].values)
)
val_ds = (
    val_ds
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Label mapping:", label_map)
print(f"Training images: {len(train_df)}")
print(f"Validation images: {len(val_df)}")


In [ ]:
# Build the CNN architecture
model = Sequential([
    Rescaling(1.0 / 255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# Train the model
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


In [ ]:
# Plot training and validation accuracy/loss
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")

plt.tight_layout()
plt.show()


### 3. Model Evaluation & Inference

Evaluate the trained model on the held-out validation split, display a classification report and confusion matrix, and test inference on one validation image.

**Note:** Validation metrics estimate performance on this split only. They do not establish clinical accuracy or suitability for diagnosis.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_loss_eval, val_acc_eval = model.evaluate(val_ds, verbose=1)
print(f"\nValidation Loss: {val_loss_eval:.4f}")
print(f"Validation Accuracy: {val_acc_eval:.4f}")

y_true = []
y_pred = []

for images, labels in val_ds:
    probabilities = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())
    y_true.extend(labels.numpy().astype(int).tolist())

class_ids = list(range(num_classes))
target_names = [
    next(name for name, index in label_map.items() if index == class_id)
    for class_id in class_ids
]

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=class_ids,
    target_names=target_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=class_ids)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=target_names,
    yticklabels=target_names,
    cmap="Blues"
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
def predict_skin_condition(image_path):
    """Predict the class of one image using the same preprocessing as training."""
    raw = tf.io.read_file(image_path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image_batch = tf.expand_dims(image, axis=0)

    probabilities = model.predict(image_batch, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    index_to_label = {index: label for label, index in label_map.items()}
    predicted_label = index_to_label[predicted_index]
    confidence = float(probabilities[predicted_index])

    plt.figure(figsize=(5, 5))
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(f"Prediction: {predicted_label} ({confidence * 100:.2f}%)")
    plt.axis("off")
    plt.show()

    return predicted_label, confidence

# Test inference on one validation sample
sample_img_path = val_df["path"].iloc[0]
sample_prediction = predict_skin_condition(sample_img_path)
print("Predicted class and confidence:", sample_prediction)


In [ ]:
# Print metrics from this training run (not pre-filled example values)
print(
    f"Final Training Accuracy: "
    f"{history.history['accuracy'][-1] * 100:.2f}%"
)
print(
    f"Final Validation Accuracy: "
    f"{history.history['val_accuracy'][-1] * 100:.2f}%"
)
print(f"Validation Loss: {val_loss_eval:.4f}")


### Final Accuracy Summary

The training and validation accuracy values below are calculated from the current run. They will update whenever the notebook is trained again.

The validation accuracy is an experimental metric for this dataset split. It must not be interpreted as diagnostic accuracy or evidence that the model can safely identify skin cancer.

### Detailed Performance Results

The classification report and confusion matrix above provide class-wise precision, recall, F1-score, and support for the current validation split.

**Interpretation reminders**
- A small difference between training and validation accuracy does not, by itself, prove that a model generalizes robustly.
- Class imbalance can make overall accuracy look high while minority-class performance remains poor.
- Check the class-wise recall and support values before drawing conclusions.
- The filename-based labels in this project must be checked against reliable dataset labels.
- This project is an educational prototype, not a medical diagnostic system.

In [ ]:
# Save the trained model package for the Streamlit application.
# Keep this architecture/configuration and label mapping format in sync with app.py.
import joblib

model_data = {
    "weights": model.get_weights(),
    "config": model.get_config(),
    "label_map": label_map,
    "final_val_accuracy": float(val_acc_eval),
    "final_val_loss": float(val_loss_eval),
}

model_filename = "skin_disease_cnn_model.joblib"
joblib.dump(model_data, model_filename)
print(f"Saved model package: {model_filename}")

# Download the model file when running in Google Colab.
try:
    from google.colab import files
    files.download(model_filename)
except ImportError:
    print("Not running in Google Colab; model file is saved in the current working directory.")
